4. Risk Labeling

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()


import pandas as pd
IN, OUT, REPORT = ROOT/'data/processed/features.csv', ROOT/'data/processed', ROOT/'results/reports'
t = pd.read_csv(IN)
# Primary outcome is recorded by the source system. So, it does not require assumptions.
t['overdue_label'] = t['overdue'].astype(int)
# Optional operational triage label, clearly separated from the supervised target.
t['risk_band'] = pd.cut(t['overdue_label'], bins=[-1,0,1], labels=['Low risk','High risk'])
leakage = ['overdue','status','report_status','status_changed','days_open','issues_per_project','risk_band','ref']
final = t.drop(columns=[c for c in leakage if c in t], errors='ignore')
final.to_csv(OUT/'final_dataset.csv', index=False)
label_summary = pd.DataFrame({'count': t['overdue_label'].value_counts(), 'share': t['overdue_label'].value_counts(normalize=True)}).rename_axis('overdue_label')
label_summary.to_csv(REPORT/'label_distribution.csv')
print('Primary research question: Can task metadata available when a task is raised identify tasks at elevated overdue risk?')
print('Class imbalance:', label_summary.to_dict())
label_summary


Primary research question: Can task metadata available when a task is raised identify tasks at elevated overdue risk?
Class imbalance: {'count': {0: 11566, 1: 858}, 'share': {0: 0.9309401159047006, 1: 0.06905988409529942}}


,count,share
overdue_label,,
0,11566,0.93094
1,858,0.06906


Due to the fact that the classes are imbalanced (about 93% non-overdue and about 7% overdue), we need to implement some solutions in order to prevent the model bias toward the majority class.